[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/19_pii_scanner_redaction.ipynb)

# 🔍 PII Scanner & Redaction — Governed Healthcare Pipeline

A healthcare support agent that accesses patient records. AIRG scans every tool call for PII
(SSNs, emails, credit cards, phone numbers) and automatically boosts risk scores to prevent data leakage.

**What you'll learn:**
- Scan tool arguments for PII entities with confidence scores
- See how PII detection integrates into the risk evaluation pipeline
- Field-level detection across nested tool arguments
- Real-world: preventing a support bot from emailing patient SSNs

**Setup:** Set `GOVERNOR_API_KEY` in Colab Secrets (🔑 icon) or as an environment variable.

## 1. Install & Configure

In [ ]:
!pip install -q httpx

import os, httpx

# Colab Secrets or environment variable
try:
    from google.colab import userdata
    API_KEY = userdata.get("GOVERNOR_API_KEY")
except Exception:
    API_KEY = os.getenv("GOVERNOR_API_KEY", "YOUR_KEY")

BASE = os.getenv("GOVERNOR_URL", "https://api.airg.nov-tia.com")
headers = {"X-API-Key": API_KEY}
print(f"✅ Connected to {BASE}")

## 2. Scenario — Support Bot Tries to Email Patient Data

In [ ]:
# A healthcare bot is about to email patient info — scan for PII first
r = httpx.post(f"{BASE}/pii/scan", headers=headers, json={
    "tool": "send_email",
    "args": {
        "to": "billing@hospital.com",
        "subject": "Patient Record Transfer",
        "body": (
            "Patient: Jane Smith
"
            "SSN: 987-65-4321
"
            "DOB: 03/15/1985
"
            "Insurance ID: BCBS-44112233
"
            "Email: jane.smith@gmail.com
"
            "Phone: (555) 867-5309
"
            "CC on file: 4111-1111-1111-1111"
        ),
    },
})
scan = r.json()

print(f"🔍 PII Detected: {scan['has_pii']}")
print(f"📊 Total findings: {scan['total_findings']}")
print(f"⚡ Risk boost: +{scan['risk_boost']} points
")

for f in scan["input_scan"]["findings"]:
    print(f"  📌 {f['entity_type']:20s} | {f['value_redacted']}")
    print(f"     Confidence: {f['confidence']}  Field: {f['field_path']}")

## 3. PII Boosts Risk in the Evaluation Pipeline

In [ ]:
# Now evaluate the same action through the full governance pipeline
",
# The PII detection automatically boosts the risk score
r = httpx.post(f"{BASE}/actions/evaluate", headers=headers, json={
    "agent_id": "healthcare-bot",
    "tool": "send_email",
    "args": {"body": "Patient SSN: 987-65-4321, CC: 4111-1111-1111-1111"},
})
d = r.json()

print(f"Decision  : {d['decision']}")
print(f"Risk score: {d['risk_score']}/100  (boosted by PII)")
print(f"Reason    : {d['explanation']}")

## 4. Clean Data — No PII Detected

In [ ]:
# Compare with a clean message — no PII, no risk boost
r = httpx.post(f"{BASE}/pii/scan", headers=headers, json={
    "tool": "send_email",
    "args": {
        "to": "team@hospital.com",
        "body": "Reminder: Staff meeting at 3pm in Conference Room B.",
    },
})
clean = r.json()

print(f"PII Detected: {clean['has_pii']}")
print(f"Findings    : {clean['total_findings']}")
print(f"Risk boost  : +{clean['risk_boost']} points")
print("
✅ Clean text passes through with zero PII risk.")

## 5. Batch Scan — Multiple Messages

In [ ]:
# Scan multiple messages to find which ones contain PII
messages = [
    {"tool": "chat", "args": {"text": "Hello, how can I help you?"}},
    {"tool": "chat", "args": {"text": "Your account 4532-1111-2222-3333 is active"}},
    {"tool": "chat", "args": {"text": "Call us at (800) 555-0199"}},
    {"tool": "chat", "args": {"text": "The weather is nice today"}},
]

print("Batch PII Scan Results:")
print(f"{"─"*50}")
for i, msg in enumerate(messages):
    r = httpx.post(f"{BASE}/pii/scan", headers=headers, json=msg)
    s = r.json()
    icon = "🛑" if s["has_pii"] else "✅"
    print(f"  {icon} Message {i+1}: {s['total_findings']} findings, "
          f"risk_boost=+{s['risk_boost']}")

## Summary

| Feature | What It Does |
|---|---|
| **PII Scan** | Detects SSNs, emails, credit cards, phone numbers |
| **Field-level** | Pinpoints which field contains PII |
| **Risk Boost** | Auto-increases risk score when PII found |
| **Pipeline Integration** | PII findings flow into evaluate() decisions |

→ [Full API docs](https://api.airg.nov-tia.com/docs) · [More recipes](https://github.com/othnielObasi/airg-cookbook)